In [1]:
import os

In [2]:
%pwd

'/Users/harshpatel/Desktop/Projects/End-to-End-Kidney-Disease-Classification-Deep-Learning-Project/notebooks'

In [3]:
os.chdir('../')

In [4]:
%pwd

'/Users/harshpatel/Desktop/Projects/End-to-End-Kidney-Disease-Classification-Deep-Learning-Project'

In [5]:
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class PreapreBaseModelConfig:
    root_dir: Path
    base_model_path: Path
    updated_base_model_path: Path
    params_image_size: list
    params_learing_rate: float
    params_include_top: bool
    params_weights: str
    params_classes: int

In [6]:
from KidneyDiseaseClassification.constants import *
from KidneyDiseaseClassification.utils.common import read_yaml,create_directories
from KidneyDiseaseClassification.utils.exception import CustomException
from KidneyDiseaseClassification.utils.logger import logger
import sys

In [ ]:
class ConfigurationManager:
    def __init__(
        self,
        config_filepath = CONFIG_FILE_PATH,
        params_filepath = PARAMS_FILE_PATH):

        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)

        create_directories([self.config.artifacts_root])

    def prepare_base_model_config(self) -> PreapreBaseModelConfig:
        config = self.config.prepare_base_model

        create_directories([config.root_dir])

        prepare_base_model_config = PreapreBaseModelConfig(
            root_dir= config.root_dir,
            base_model_path= config.base_model_path,
            updated_base_model_path= config.update_base_model,
            params_image_size= self.params.IMAGE_SIZE,
            params_learing_rate = self.params.LEARNING_RATE,
            params_include_top=  self.params.INCLUDE_TOP,
            params_weights= self.params.WEIGHTS,
            params_classes=self.params.CLASSES

        )

        return prepare_base_model_config

In [8]:
import os
import urllib.request as request
from zipfile import ZipFile
import tensorflow as tf

In [ ]:
class PrepareBaseModel:
    def __init__(self,config:PreapreBaseModelConfig):
        self.config = config

    def get_base_model(self):
        self.model = tf.keras.applications.EfficientNetB0(
            input_shape = self.config.params_image_size,
            weights = self.config.params_weights,
            include_top = self.config.params_include_top
        )

        self.save_model(path=self.config.base_model_path,model=self.model)


    @staticmethod
    def _prepare_full_model(model, classes, freeze_all, freeze_till, learning_rate):
        if freeze_all:
            for layer in model.layers:
                layer.trainable = False        
        elif (freeze_till is not None) and (freeze_till > 0):
            for layer in model.layers[:-freeze_till]:
                layer.trainable = False        

        flatten_in = tf.keras.layers.Flatten()(model.output)
        dense = tf.keras.layers.Dense(units=256, activation='relu')(flatten_in)   
        dropout = tf.keras.layers.Dropout(0.5)(dense)                             
        prediction = tf.keras.layers.Dense(units=classes, activation='softmax')(dropout)

        full_model = tf.keras.models.Model(
            inputs=model.input,
            outputs=prediction
        )

        full_model.compile(
            optimizer=tf.keras.optimizers.Adam(learning_rate=learning_rate),  
            loss=tf.keras.losses.CategoricalCrossentropy(),                   
            metrics=['accuracy']
        )

        full_model.summary()
        return full_model
        
    def update_base_model(self):
        self.full_model = self._prepare_full_model(
            model=self.model,
            classes=self.config.params_classes,
            freeze_all=True,
            freeze_till=None,
            learning_rate=self.config.params_learing_rate
        )

        self.save_model(path=self.config.updated_base_model_path , model=self.full_model)

    @staticmethod
    def save_model(path : Path  , model: tf.keras.Model):
        model.save(path)


In [10]:
try:
    config = ConfigurationManager()
    prepare_base_model_config = config.prepare_base_model_config()
    prepare_base_model = PrepareBaseModel(config=prepare_base_model_config)
    prepare_base_model.get_base_model()
    prepare_base_model.update_base_model()
except Exception as e:
    raise CustomException(e,sys)
    

[2026-06-04 00:17:06,859: INFO: common: yaml file: config/config.yaml loaded successfully]
[2026-06-04 00:17:06,861: INFO: common: yaml file: params.yaml loaded successfully]
[2026-06-04 00:17:06,862: INFO: common: created directory at: artifacts]
[2026-06-04 00:17:06,862: INFO: common: created directory at: artifacts/prepare_base_model]
[2026-06-04 00:17:07,124: WARNING: saving_utils: Compiled the loaded model, but the compiled metrics have yet to be built. `model.compile_metrics` will be empty until you train or evaluate the model.]
[2026-06-04 00:17:07,184: WARNING: optimizer: At this time, the v2.11+ optimizer `tf.keras.optimizers.SGD` runs slowly on M1/M2 Macs, please use the legacy Keras optimizer instead, located at `tf.keras.optimizers.legacy.SGD`.]
Model: "model"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 input_1 (InputLayer)        [(None, 224, 224, 3)]     0         
                   

/opt/miniconda3/envs/kidney/lib/python3.10/site-packages/keras/src/engine/training.py:3103: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(
